In [ ]:
import sys
from scripts.tools.music_info_extractor import MusicInfoExtractor

sys.path.append("../../")

In [16]:
import os

from scripts.data.datasets import get_audio_filenames

dir = "/home/gjzhao/downloads/fma_large"
files = get_audio_filenames([dir])
len(files)

106573

In [18]:
from scripts.factory import load_model
from scripts.models.audio_autoencoder import AudioAutoencoder

ckpt_path = "/home/gjzhao/workspace2024/heartstrings-version4/ckpt/vae48000.ckpt"
model: AudioAutoencoder = load_model(ckpt_path)
model.cuda()

AudioAutoencoder(
  (bottleneck): VAEBottleneck()
  (encoder): OobleckEncoder(
    (layers): Sequential(
      (0): Conv1d(2, 150, kernel_size=(7,), stride=(1,), padding=(3,))
      (1): EncoderBlock(
        (layers): Sequential(
          (0): ResidualUnit(
            (layers): Sequential(
              (0): SnakeBeta()
              (1): Conv1d(150, 150, kernel_size=(7,), stride=(1,), padding=(3,))
              (2): SnakeBeta()
              (3): Conv1d(150, 150, kernel_size=(1,), stride=(1,))
            )
          )
          (1): ResidualUnit(
            (layers): Sequential(
              (0): SnakeBeta()
              (1): Conv1d(150, 150, kernel_size=(7,), stride=(1,), padding=(9,), dilation=(3,))
              (2): SnakeBeta()
              (3): Conv1d(150, 150, kernel_size=(1,), stride=(1,))
            )
          )
          (2): ResidualUnit(
            (layers): Sequential(
              (0): SnakeBeta()
              (1): Conv1d(150, 150, kernel_size=(7,), stride=(

In [20]:
from scripts.tools.music_info_extractor import MusicInfoExtractor
from tqdm import tqdm
import torch


def process(files):
    mie = MusicInfoExtractor(sample_rate=48000)
    target_dir = "/home/gjzhao/data/training_data/latent_48000"
    os.makedirs(target_dir, exist_ok=True)
    for file in tqdm(files):
        basename = os.path.basename(file)
        target_path = f"{target_dir}/{basename}.pt"
        if os.path.exists(target_path):
            continue
        mie.file_path = file
        wav = mie.demix(track_name="no_vocals")
        with torch.no_grad():
            latent = model.encode_audio(wav.unsqueeze(0).cuda(), chunked=True)
            info = {"latent": latent[0].cpu(), "file": file}
            torch.save(info, target_path)

  0%|          | 85/106573 [01:32<32:19:03,  1.09s/it]


KeyboardInterrupt: 

In [4]:
wav.shape

torch.Size([2, 14440211])

In [5]:
import torchaudio

torchaudio.save("test.wav", wav, 48000)

In [6]:
import soundfile as sf

try:
    audio_data, sample_rate = sf.read(file)
    print(f"采样率: {sample_rate}")
    print(f"音频数据形状: {audio_data.shape}")
except RuntimeError as e:
    print(f"读取文件时出错: {e}")

读取文件时出错: Error opening '/home/gjzhao/data/music/纯音乐 200 首/200 - 200.友谊天长地久.mp3': File contains data in an unknown format.


In [ ]:
from scripts.factory import load_model
from scripts.models.audio_autoencoder import AudioAutoencoder

ckpt_path = "/home/gjzhao/workspace2024/heartstrings-version4/ckpt/vae48000.ckpt"
model: AudioAutoencoder = load_model(ckpt_path)

/home/gjzhao/miniconda3/envs/heartstrings/lib/python3.10/site-packages/vector_quantize_pytorch/vector_quantize_pytorch.py:461: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/home/gjzhao/miniconda3/envs/heartstrings/lib/python3.10/site-packages/vector_quantize_pytorch/vector_quantize_pytorch.py:674: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/home/gjzhao/miniconda3/envs/heartstrings/lib/python3.10/site-packages/vector_quantize_pytorch/finite_scalar_quantization.py:162: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


In [11]:
import os
import torch

wav = wav.cuda()
model.cuda()
with torch.no_grad():
    latent = model.encode_audio(wav.unsqueeze(0), chunked=True)
    recon = model.decode_audio(latent, chunked=True)

In [12]:
torchaudio.save("recon.wav", recon.squeeze(0).cpu(), 48000)
torchaudio.save("origin.wav", wav.cpu(), 48000)

In [4]:
import sys

sys.path.append("../../")
from scripts.tools.music_info_extractor import MusicInfoExtractor


mie = MusicInfoExtractor(sample_rate=48000, device="cuda:1")

/home/gjzhao/miniconda3/envs/heartstrings/lib/python3.10/site-packages/natten/functional.py:83: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(
/home/gjzhao/miniconda3/envs/heartstrings/lib/python3.10/site-packages/natten/functional.py:205: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(
/home/gjzhao/miniconda3/envs/heartstrings/lib/python3.10/site-packages/natten/functional.py:261: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(
/home/gjzhao/miniconda3/envs/heartstrings/lib/python3.10/site-packages/natten/functional.py:381: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backw

In [5]:
file = "/home/gjzhao/workspace2024/heartstrings-version4/test_data/38BEETS - さくらの季節 (1).mp3"
mie.file_path = file
wav = mie.demix(track_name="no_vocals")

In [6]:
wav = wav.numpy()
wav.shape

(2, 7873149)

In [ ]:
import librosa

onset_env = librosa.onset.onset_strength(y=wav, sr=48000)

In [10]:
onset_env.shape

(2, 15378)

In [ ]:
onset_env.max(), onset_env.min()

(10.484766, 0.0)

In [ ]:
import numpy as np

chroma = librosa.feature.chroma_stft(
    S=np.abs(librosa.stft(wav, hop_length=4096 * 2)), sr=48000
)

In [ ]:
chroma.shape, chroma.max(), chroma.min()

((2, 12, 962), 1.0, 0.019793924)

In [13]:
import torch.nn.functional as F
import torch

chroma2 = F.interpolate(torch.from_numpy(chroma), (300,))

In [14]:
chroma2.shape

torch.Size([2, 12, 300])

In [15]:
import torch 
file= "/home/gjzhao/data/training_data/latent_48000_chroma/-95456.mp3.pt"
data = torch.load(file )

/tmp/ipykernel_307469/1678721934.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file )


In [18]:
data['onset_env'].shape

torch.Size([2, 751])

In [20]:
data['latent'].shape

torch.Size([64, 751])